# Sentiment Analysis

### Neural Bag of Words

In [4]:
import collections
import numpy as np
import torch 
import torch.nn as nn
import torch.optim as optim
#import torchtext
#pip install torchtext
import tqdm 
import os 

In [5]:
seed = 42
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.backends.cudnn.deterministic = True

In [6]:
print(os.path.abspath("arg_mining/ml_algorithms/ML/datasets/test.conll"))

c:\Users\huonc\Desktop\python\mining_project\arg_mining\ml_algorithms\ML\arg_mining\ml_algorithms\ML\datasets\test.conll


In [7]:
train_path = "datasets/train.conll"
test_path = "datasets/test.conll"
with open(train_path, encoding="utf-8") as f:
    train_data = f.read()

with open(test_path, encoding = "utf-8") as f:
    test_data = f.read()

In [8]:
#extract first column 
def get_first_column(data):
    lines = data.strip().split('\n')
    first_col = []

    for i in lines:
        if i.strip():
            cols = i.split('\t')
            first_col.append(cols[0])

    return first_col

train_data = get_first_column(train_data)
test_data = get_first_column(test_data)

len(train_data), len(test_data)


(942743, 237236)

In [9]:
#vocab to keep
negation_words = [
    "not", "no", "never", "none", "nothing", "neither", "nor",
    "hardly", "scarcely", "barely", "without"
]
intensifiers = [
    "very", "really", "extremely", "quite", "so", "too", "just",
    "absolutely", "totally", "incredibly", "barely", "fairly", "almost", "nearly"
]
modal_verbs = [
    "could", "would", "should", "might", "may", "must", "can", "shall", "will"
]
auxiliary_verbs = [
    "is", "are", "was", "were", "be", "been", "being", "am",
    "do", "does", "did", "have", "has", "had"
]
pronouns = [
    "i", "you", "we", "they", "he", "she", "it",
    "me", "us", "them", "my", "your", "our", "their",
    "mine", "yours", "his", "hers", "its"
]
conjunctions = [
    "but", "although", "though", "yet", "while", "whereas"
]
subjective_adverbs = [
    "always", "never", "sometimes", "often", "seldom",
    "unfortunately", "fortunately", "luckily", "sadly", "happily"
]
exception_words = (
    negation_words +
    intensifiers +
    modal_verbs +
    auxiliary_verbs +
    pronouns +
    conjunctions +
    subjective_adverbs
)
exception_words[:10]

['not',
 'no',
 'never',
 'none',
 'nothing',
 'neither',
 'nor',
 'hardly',
 'scarcely',
 'barely']

In [10]:
'''
open the conll file then remove whitespace, tabs, newline
there is only 1 column in the conll files so it'll just append those words into current_sentence
then join the current word to another word below to make a sentence
the final else indicates the last word of a sentence
'''
def read_conll_file(file_path):
    sentences = []
    current_sentence = []
    with open(file_path, encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                parts = line.split('\t')
                word = parts[0]
                current_sentence.append(word)
            else:
                if current_sentence:
                    sentences.append(' '.join(current_sentence))
                    current_sentence = []

    if current_sentence:
        sentences.append(' '.join(current_sentence))
    return sentences

In [11]:
train = read_conll_file("datasets/train.conll")
train

['14 shelwood46',
 'Comment Florida',
 '16 mortemdeus',
 '35 TTRoadHog',
 'No you probably dont want a lawyer who has a facial tattoo saying bitch but I also dont care about someone having a couple of visible pieces even in professionals But thats just my own sensibility others have their own',
 'But indulge my thought process for a moment',
 '97 HotSteak',
 'I was constantly losing my tie or forgetting to wear a belt with my pants and potentially getting in trouble for being out of uniform Any idea that it eliminates dress code violations is a myth',
 'Ive personally worked in jobs where anonymous client surveys can 100 lead to termination and those were just banking jobs The stress is genuine',
 'Comment My school would get Jewish holidays off despite the fact that there were no Jewish kids there most of the school was black some Hispanic and only one Jewish teacher',
 'Comment Florida Mexico Bahamas Dominican Republic Virgin Islands and Hawaii',
 'Comment In terms of pay yes',
 'Tha